# Hands-on: Monthly Anomalies and Arctic Averaging

Follow-up to YL's 20-minute Binder demonstration. The default data are synthetic. EG's exact reference uses 1981–2010 and an **unweighted** Arctic mean.

Run all cells to see worked solutions, or pause at each question before expanding the solution.

In [ ]:
from pathlib import Path
import os
import subprocess
import numpy as np
import xarray as xr
if Path.cwd().name == 'notebooks':
    os.chdir('..')
DATA = Path(os.environ.get('WORKSHOP_INPUT', 'data/demo/tas_demo.nc'))
OUT = Path('outputs/hands_on_monthly')
OUT.mkdir(parents=True, exist_ok=True)
with xr.open_dataset(DATA) as ds:
    tas = (ds.tas - 273.15).load()
print('Input:', DATA)

## Exercise 1: Why group by calendar month?

Predict how subtracting one scalar annual baseline would differ from subtracting 12 calendar-month means. Check that the selected baseline has 360 monthly values and that the resulting monthly anomalies average to zero for every baseline month.

In [ ]:
ref = tas.sel(time=slice('1981', '2010'))
assert ref.sizes['time'] == 360
clim = ref.groupby('time.month').mean('time')
anom = tas.groupby('time.month') - clim
residual = anom.sel(time=slice('1981', '2010')).groupby('time.month').mean('time')
print('Largest baseline residual:', float(abs(residual).max()), 'degC')
assert float(abs(residual).max()) < 3e-4

## Exercise 2: Change the baseline to 1981–2000

Compute the new climatology independently with CDO and xarray. Compare anomalies on 1979–2014. Keep this exercise output separate from the source-matching workflow.

In [ ]:
subprocess.run(['cdo', '-O', '-L', 'subc,273.15', str(DATA), str(OUT / 'celsius.nc')], check=True)
subprocess.run(['cdo', '-O', '-L', 'ymonmean', '-selyear,1981/2000', str(OUT / 'celsius.nc'), str(OUT / 'climatology_1981_2000.nc')], check=True)
subprocess.run(['cdo', '-O', '-L', 'ymonsub', str(OUT / 'celsius.nc'), str(OUT / 'climatology_1981_2000.nc'), str(OUT / 'anomaly_full.nc')], check=True)
subprocess.run(['cdo', '-O', '-L', 'seldate,1979-01-01,2014-12-31', str(OUT / 'anomaly_full.nc'), str(OUT / 'anomaly_1979_2014.nc')], check=True)
py_new = tas.groupby('time.month') - tas.sel(time=slice('1981', '2000')).groupby('time.month').mean('time')
py_new = py_new.sel(time=slice('1979', '2014'))
with xr.open_dataset(OUT / 'anomaly_1979_2014.nc') as ds:
    difference = float(abs(ds.tas - py_new).max())
print('New-baseline comparison:', difference, 'degC')
assert difference < 3e-4

## Exercise 3: Arithmetic versus area-weighted Arctic means

Predict which parts of a regular latitude–longitude grid receive too much weight in an arithmetic mean. Calculate both definitions. EG's reference uses the arithmetic version; area weighting is an intentional extension.

In [ ]:
region = anom.sel(time=slice('1979', '2014')).where((anom.lat >= 65) & (anom.lat <= 90), drop=True)
arithmetic = region.mean(('lat', 'lon'))
weighted = region.weighted(np.cos(np.deg2rad(region.lat))).mean(('lat', 'lon'))
print('Largest change caused by the scientific definition:', float(abs(arithmetic-weighted).max()), 'degC')
print('Both are meaningful, but they answer different averaging questions.')

## Exercise 4: Explain the colours

EG's annual stripes normalize the 36 annual Arctic anomalies using the largest absolute annual anomaly. Identify the range and the years with the largest positive/negative anomalies. Changing the baseline can change both the values and the data-derived colour range.

In [ ]:
annual = arithmetic.groupby('time.year').mean('time')
limit = float(abs(annual).max())
print('Stripes:', annual.size)
print(f'Colour range: {-limit:.4f} to {limit:.4f} degC')
print('Most negative year:', int(annual.idxmin()))
print('Most positive year:', int(annual.idxmax()))
assert annual.size == 36

## Save your work

Download the modified notebook and any output files before closing Binder. For full-grid comparisons with the other presenters, use the same ACCESS-CM2 input file, baseline, selected period, averaging weights and colour normalization.